# Global directed causal relations

Extract cause–relation–effect frames from all available policy and commentary sentences. Nodes correspond to the normalized cause/effect phrases themselves; no NMF assignments are required.

**Interpretation:** The directed graph represents linguistic claims, not verified causal mechanisms.


## Prepare analysis environment
Use the retained extraction and figure utilities.


In [1]:
# Prepare a global relation-graph comparison using global sentence evidence.
# Extraction quality checks and network metrics are kept at their original settings.

import html
import io
import os

os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")
import re
import shutil
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import numpy as np
import pandas as pd
from PIL import Image
from scipy.spatial.distance import jensenshannon
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

pd.set_option("display.max_columns", 180)
pd.set_option("display.width", 260)

RANDOM_STATE = 42
MIN_COUNTRY_ITEMS = 10
MIN_COUNTRY_SOURCES = 2
MIN_FRAME_QUALITY = 0.40
MAX_SENTENCE_WORDS = 120
MIN_SPAN_WORDS = 2
MAX_SPAN_WORDS = 55
BOOTSTRAP_ITERATIONS = 250

BLUE = "#1f77b4"
ORANGE = "#ff7f0e"
GAP_CMAP = LinearSegmentedColormap.from_list(
    "sentiment_to_policy_gap",
    [ORANGE, "#ffffff", BLUE],
)


def find_project_root() -> Path:
    """Locate the project by its combined source-segment dataset."""
    candidates = []
    configured = os.environ.get("CAUSAL_NLP_PROJECT_ROOT")
    if configured:
        candidates.append(Path(configured).expanduser().resolve())
    cwd = Path.cwd().resolve()
    candidates.extend([cwd, *cwd.parents])
    for candidate in candidates:
        if (candidate / "data" / "policy_feedback_segments.csv").is_file():
            return candidate
    raise FileNotFoundError("Run from the extracted project or set CAUSAL_NLP_PROJECT_ROOT.")

PROJECT_ROOT = find_project_root()
CSVS_CHUNKED = PROJECT_ROOT / "data"
CHUNKS_ALL = CSVS_CHUNKED / "policy_feedback_segments.csv"
METHOD_ROOT = Path(
    os.environ.get(
        "CAUSAL_NLP_OUTPUT_ROOT",
        PROJECT_ROOT / "causal_semantics",
    )
).expanduser().resolve()



# Shared causal-text cleaning layer
import sys

_SHARED_CANDIDATES = [
    Path(os.environ.get("CAUSAL_NLP_SHARED_MODULE_DIR", "")).expanduser(),
    PROJECT_ROOT / "causal_semantics",
    Path.cwd() / "shared",
    Path.cwd(),
]
SHARED_MODULE_DIR = next(
    (
        candidate.resolve()
        for candidate in _SHARED_CANDIDATES
        if str(candidate) and (candidate / "causal_text_cleaning.py").exists()
    ),
    None,
)
if SHARED_MODULE_DIR is None:
    raise FileNotFoundError(
        "causal_text_cleaning.py was not found. Place it in "
        "causal_semantics or set CAUSAL_NLP_SHARED_MODULE_DIR."
    )
if str(SHARED_MODULE_DIR) not in sys.path:
    sys.path.insert(0, str(SHARED_MODULE_DIR))

from causal_text_cleaning import (
    CAUSAL_ARTIFACT_PHRASES,
    CAUSAL_ARTIFACT_TOKENS,
    CAUSAL_EMBEDDING_STOPWORDS,
    CLEANING_VERSION,
    build_context_windows,
    contains_contact_or_link,
    content_token_ratio,
    infer_sentiment_country,
    is_source_residue,
    load_or_build_clean_sentence_inventory,
    normalize_source_text,
    sentence_units,
)

# Compatibility names used by the method-specific extraction code.
normalize_for_noise_check = normalize_source_text
split_sentences = sentence_units


def normalise_span(text: str) -> str:
    return re.sub(r"[^a-z0-9à-ÿ]+", " ", str(text).lower()).strip()


def normalise_minmax(series: pd.Series) -> pd.Series:
    values = pd.to_numeric(series, errors="coerce").fillna(0.0)
    minimum = float(values.min())
    maximum = float(values.max())
    if maximum <= minimum:
        return pd.Series(np.zeros(len(values)), index=series.index)
    return (values - minimum) / (maximum - minimum)


def save_verified_png(fig, output_path: Path, dpi: int = 160) -> None:
    """Write a standard RGB PNG directly to the final path and verify it."""
    output_path = Path(output_path).expanduser().resolve()
    output_path.parent.mkdir(parents=True, exist_ok=True)
    raw_buffer = io.BytesIO()
    rgb_buffer = io.BytesIO()
    try:
        fig.savefig(
            raw_buffer,
            format="png",
            dpi=dpi,
            bbox_inches="tight",
            facecolor="white",
            transparent=False,
        )
        plt.close(fig)
        raw_buffer.seek(0)
        with Image.open(raw_buffer) as image:
            image.load()
            image.convert("RGB").save(
                rgb_buffer,
                format="PNG",
                optimize=False,
                compress_level=6,
            )
        png_bytes = rgb_buffer.getvalue()
        if not png_bytes.startswith(b"\x89PNG\r\n\x1a\n") or len(png_bytes) <= 100:
            raise OSError("Invalid PNG bytes were generated.")
        output_path.unlink(missing_ok=True)
        output_path.write_bytes(png_bytes)
        with Image.open(output_path) as image:
            image.load()
            if image.format != "PNG" or image.mode != "RGB":
                raise OSError(f"Invalid final PNG: {output_path}")
            width, height = image.size
        print("Saved verified PNG:", output_path, f"size=({width}, {height})")
    finally:
        plt.close(fig)
        raw_buffer.close()
        rgb_buffer.close()

# Map public metadata labels to the original analytical interface.
SOURCE_SCHEMA = {'record_ref': 'doc_id', 'source_label': 'filename', 'source_path': 'source_file', 'relative_path': 'relative_source_file', 'collection': 'corpus', 'origin_type': 'source_type', 'augmentation_type': 'synthetic_type', 'geography': 'country', 'document_characters': 'char_count', 'document_words': 'word_count', 'segment_key': 'chunk_id', 'segment_index': 'chunk_index', 'section_context': 'heading_context', 'segment_content': 'chunk_text', 'segment_characters': 'chunk_char_count', 'segment_words': 'chunk_word_count'}


## Load pooled sentence evidence
Attach original text context directly to each cleaned sentence.


In [2]:
# Create output folders and reuse the prepared sentence inventory.
OUTPUT_DIR = METHOD_ROOT / "output" / "relation_graphs"
IMG_DIR = METHOD_ROOT / "figures" / "relation_graphs"
SHARED_OUTPUT_DIR = METHOD_ROOT / "output" / "shared"
for folder in (OUTPUT_DIR, IMG_DIR, SHARED_OUTPUT_DIR):
    folder.mkdir(parents=True, exist_ok=True)

CLEAN_SENTENCE_INVENTORY = SHARED_OUTPUT_DIR / "sentence_evidence_cache.csv"
CLEAN_SENTENCE_METADATA = SHARED_OUTPUT_DIR / "sentence_evidence_metadata.json"
CLEAN_SENTENCE_SUMMARY = SHARED_OUTPUT_DIR / "sentence_evidence_summary.csv"
CLEAN_SENTENCE_VALIDATION = SHARED_OUTPUT_DIR / "sentence_evidence_validation.csv"

clean_sentence_inventory, shared_cleaning_metadata = load_or_build_clean_sentence_inventory(
    CHUNKS_ALL, CLEAN_SENTENCE_INVENTORY, CLEAN_SENTENCE_METADATA,
    CLEAN_SENTENCE_SUMMARY, CLEAN_SENTENCE_VALIDATION,
)
raw_chunks_all = build_context_windows(pd.read_csv(CHUNKS_ALL).rename(columns=SOURCE_SCHEMA))
context_columns = raw_chunks_all[["chunk_id", "chunk_text", "context_window"]].drop_duplicates("chunk_id")

def select_sentences(corpus: str, source_type: str) -> pd.DataFrame:
    # Join sentence-level evidence directly to its source segment; do not filter by topics.
    selected = clean_sentence_inventory[
        clean_sentence_inventory["corpus"].eq(corpus)
        & clean_sentence_inventory["source_type"].eq(source_type)
    ].copy().merge(context_columns, on="chunk_id", how="left", validate="many_to_one")
    selected["original_chunk_text"] = selected["chunk_text"]
    selected["chunk_text"] = selected["clean_sentence"]
    selected["analysis_country"] = "global"
    return selected

policy_chunks = select_sentences("policy", "original")
sentiment_chunks = select_sentences("sentiment", "original")
synthetic_chunks = select_sentences("sentiment", "synthetic")

inventory_coverage = pd.DataFrame([
    {"source_set": label, "sentences": len(frame), "source_segments": frame["chunk_id"].nunique(),
     "documents": frame["doc_id"].nunique()}
    for label, frame in (("policy", policy_chunks), ("commentary", sentiment_chunks),
                         ("synthetic_probe", synthetic_chunks))
])
inventory_coverage.to_csv(OUTPUT_DIR / "global_relation_sentence_inputs.csv", index=False)
print(inventory_coverage.to_string(index=False))


     source_set  sentences  source_segments  documents
         policy      15281             1930         56
     commentary       4182              522         17
synthetic_probe       2128              212         53


## Extract causal frames
Keep the original multilingual cue definitions, relation classes, and quality filters.


In [3]:
# --- Analysis stage ---------------------------------------------------------


def cue_regex(cues: str) -> re.Pattern:
    return re.compile(rf"(?<!\w)(?:{cues})(?!\w)", flags=re.IGNORECASE)


MODAL = r"(?:to|can|could|may|might|must|should|will|would|shall)"

FRAME_CUES = [
    (
        cue_regex(
            r"because of|because|due to|owing to|as a result of|resulting from|"
            r"caused by|driven by|triggered by"
        ),
        "causes_or_increases",
        "reverse",
        1.00,
    ),
    (
        cue_regex(
            r"may lead to|can lead to|could lead to|is likely to lead to|"
            r"lead(?:s|ing)? to|gives? rise to|result(?:s|ed|ing)? in|"
            r"bring(?:s|ing)? about|brought about|caus(?:e|es|ed|ing)|"
            r"contribut(?:e|es|ed|ing) to|drive|drives|driving|"
            r"produc(?:e|es|ed|ing)|trigger(?:s|ed|ing)?|"
            rf"(?:{MODAL}\s+generate|generates|generating)|"
            rf"(?:{MODAL}\s+create|creates|creating)|"
            r"exacerbat(?:e|es|ed|ing)|worsen(?:s|ed|ing)?|"
            r"intensif(?:y|ies|ied|ying)|accelerat(?:e|es|ed|ing)|"
            r"increas(?:e|es|ed|ing)|rais(?:e|es|ed|ing)|"
            r"amplif(?:y|ies|ied|ying)|heighten(?:s|ed|ing)?"
        ),
        "causes_or_increases",
        "forward",
        1.00,
    ),
    (
        cue_regex(
            rf"(?:{MODAL}\s+reduce|reduces|reducing)|"
            rf"(?:{MODAL}\s+prevent|prevents|preventing)|"
            rf"(?:{MODAL}\s+limit|limits|limiting)|"
            rf"(?:{MODAL}\s+mitigate|mitigates|mitigating)|"
            rf"(?:{MODAL}\s+decrease|decreases|decreasing)|"
            rf"(?:{MODAL}\s+lower|lowers|lowering)|"
            r"minimi(?:s|z)(?:es|ing)|"
            r"alleviat(?:es|ing)|avoid(?:s|ing)?|curb(?:s|ing)?|"
            r"constrain(?:s|ing)?|counteract(?:s|ing)?|"
            r"protect(?:s|ing)? against|safeguard(?:s|ing)? against"
        ),
        "reduces_or_prevents",
        "forward",
        1.00,
    ),
    (
        cue_regex(
            rf"(?:{MODAL}\s+support(?:\s+and\s+assist)?|supports|supporting)|"
            rf"(?:{MODAL}\s+help(?:\s+to|\s+with|\s+in)?|"
            r"helps(?:\s+to|\s+with|\s+in)?|"
            r"helped(?:\s+to|\s+with|\s+in)?|"
            r"helping(?:\s+to|\s+with|\s+in)?)|"
            rf"(?:{MODAL}\s+enable|enables|enabling)|"
            rf"(?:{MODAL}\s+facilitate|facilitates|facilitating)|"
            rf"(?:{MODAL}\s+allow(?:\s+for)?|allows(?:\s+for)?|allowing(?:\s+for)?)|"
            r"mak(?:e|es|ing) possible|"
            r"made possible|foster(?:s|ing)?|encourag(?:e|es|ing)|"
            r"promot(?:e|es|ing)|empower(?:s|ing)?|"
            r"assist(?:s|ing)?(?:\s+with|\s+in)?|"
            r"in order to|so as to|with the aim of|for the purpose of|"
            r"with a view to|designed to|intended to"
        ),
        "enables_or_supports",
        "forward",
        0.95,
    ),
    (
        cue_regex(r"enabled by|facilitated by|made possible by"),
        "enables_or_supports",
        "reverse",
        0.95,
    ),
    (
        cue_regex(
            r"is required for|are required for|was required for|were required for|"
            r"is necessary for|are necessary for|was necessary for|were necessary for"
        ),
        "requires_or_depends_on",
        "forward",
        0.95,
    ),
    (
        cue_regex(
            r"requir(?:e|es|ing)|depend(?:s|ing)? on|"
            r"relies? on|relying on|necessitat(?:e|es|ing)|"
            r"presuppos(?:e|es|ing)|is contingent on|is conditional on"
        ),
        "requires_or_depends_on",
        "reverse",
        0.95,
    ),
    (
        cue_regex(
            r"creat(?:e|es|ed|ing) a risk of|"
            r"increas(?:e|es|ed|ing) the risk of|"
            r"rais(?:e|es|ed|ing) the risk of|"
            r"pos(?:e|es|ing) a risk to|risk(?:ed|ing)|"
            r"threaten(?:s|ed|ing)?|undermin(?:e|es|ed|ing)|"
            r"harm(?:s|ed|ing)?|jeopardi(?:s|z)(?:e|es|ed|ing)|"
            r"endanger(?:s|ed|ing)?|compromis(?:e|es|ed|ing)|"
            r"weaken(?:s|ed|ing)?"
        ),
        "risks_or_threatens",
        "forward",
        1.00,
    ),
    (
        cue_regex(
            r"is expected to improve|is intended to improve|"
            rf"(?:{MODAL}\s+improve|improves|improving)|"
            rf"(?:{MODAL}\s+enhance|enhances|enhancing)|"
            rf"(?:{MODAL}\s+strengthen|strengthens|strengthening)|"
            rf"(?:{MODAL}\s+advance|advances|advancing)|"
            r"boost(?:s|ing)?|optimi(?:s|z)(?:es|ing)"
        ),
        "expected_to_improve",
        "forward",
        0.95,
    ),
    (
        cue_regex(
            r"parce que|en raison de|à cause de|du fait de|résultant de|"
            r"causé par|causée par|provoqué par|provoquée par|"
            r"entraîné par|entraînée par|généré par|générée par"
        ),
        "causes_or_increases",
        "reverse",
        1.00,
    ),
    (
        cue_regex(
            r"peut conduire à|pourrait conduire à|est susceptible de conduire à|"
            r"conduit à|conduit au|conduit aux|donne lieu à|"
            r"entraîn(?:e|ent|ant)|caus(?:e|ent|ant)|"
            r"provoqu(?:e|ent|ant)|"
            r"engendr(?:e|ent|ant)|"
            r"contribu(?:e|ent|ant) à|"
            r"génèr(?:e|ent|ant)|déclench(?:e|ent|ant)|"
            r"accentu(?:e|ent|ant)|aggrav(?:e|ent|ant)|"
            r"accroît|accroissent|augment(?:e|ent|ant)|"
            r"amplifi(?:e|ent|ant)"
        ),
        "causes_or_increases",
        "forward",
        1.00,
    ),
    (
        cue_regex(
            r"rédui(?:t|ts|te|tes|re|sent|sant)|prévien(?:t|nent)|"
            r"limit(?:e|ent|ant)|atténu(?:e|ent|ant)|"
            r"diminu(?:e|ent|ant)|abaiss(?:e|ent|ant)|"
            r"minimis(?:e|ent|ant)|évit(?:e|ent|ant)|"
            r"frein(?:e|ent|ant)|contraint|contraignent|"
            r"protèg(?:e|ent|ant) contre"
        ),
        "reduces_or_prevents",
        "forward",
        1.00,
    ),
    (
        cue_regex(
            r"rend possible|rendent possible|permet(?:s|tent)? de|"
            r"permet(?:s|tent)?|soutien(?:t|nent)|"
            r"facilit(?:e|ent|ant)|"
            r"favoris(?:e|ent|ant)|"
            r"encourag(?:e|ent|ant)|"
            r"aid(?:e|ent|ant) à|donne les moyens de|"
            r"afin de|dans le but de|en vue de|de manière à|de façon à|"
            r"destiné à|destinée à|conçu pour|conçue pour|visant à"
        ),
        "enables_or_supports",
        "forward",
        0.95,
    ),
    (
        cue_regex(r"rendu possible par|rendue possible par"),
        "enables_or_supports",
        "reverse",
        0.95,
    ),
    (
        cue_regex(
            r"est nécessaire pour|sont nécessaires pour|"
            r"est requis pour|sont requis pour|est requise pour|sont requises pour"
        ),
        "requires_or_depends_on",
        "forward",
        0.95,
    ),
    (
        cue_regex(
            r"nécessit(?:e|es|ant)|dépend(?:s|ent)? de|"
            r"repos(?:e|es|ant) sur|exig(?:e|es|ant)|requiert|requièrent|"
            r"est tributaire de|"
            r"est conditionné par|est conditionnée par"
        ),
        "requires_or_depends_on",
        "reverse",
        0.95,
    ),
    (
        cue_regex(
            r"crée un risque de|créent un risque de|accroît le risque de|"
            r"augmente le risque de|présente un risque pour|"
            r"menac(?:e|es|é|ée|és|ées|ant)|risque de|"
            r"compromet|compromettent|nui(?:t|sent) à|met en danger|"
            r"mettent en danger|fragilis(?:e|es|é|ée|és|ées|ant)"
        ),
        "risks_or_threatens",
        "forward",
        1.00,
    ),
    (
        cue_regex(
            r"devrait améliorer|vise à améliorer|"
            r"amélior(?:e|ent|ant)|"
            r"renforc(?:e|ent|ant)|"
            r"optimis(?:e|ent|ant)"
        ),
        "expected_to_improve",
        "forward",
        0.95,
    ),
]

NOUN_PREDECESSORS = {
    "a", "an", "the", "this", "that", "these", "those", "of", "for",
    "with", "without", "under", "over", "additional", "administrative",
    "continued", "digital", "educational", "existing", "financial", "human",
    "institutional", "ongoing", "practical", "professional", "social",
    "technical", "alongside", "through", "via", "including", "concerning",
    "regarding", "around",
}

SUPPORT_NOUN_FOLLOWERS = {
    "staff", "worker", "workers", "service", "services", "team", "teams",
    "function", "functions", "material", "materials", "resource", "resources",
    "network", "networks", "group", "groups", "structure", "structures",
    "mechanism", "mechanisms", "work", "personnel", "for", "of", "from",
    "to", "and", "or",
}


def sentence_units(text: str) -> list[str]:
    cleaned = normalize_source_text(text)
    cleaned = re.sub(r"\s+[•●▪◦]\s+", ". ", cleaned)
    cleaned = re.sub(
        r"\s+-\s+(?=[A-ZÀ-ÖØ-Ý][A-Za-zÀ-ÖØ-öø-ÿ ]{2,}:)",
        ". ",
        cleaned,
    )
    cleaned = re.sub(r"\s+", " ", cleaned).strip()
    cleaned = re.sub(
        r"([.!?])(['’”\"])?\s+(?=[A-ZÀ-ÖØ-Ý])",
        lambda match: match.group(1) + (match.group(2) or "") + "\n",
        cleaned,
    )
    return [
        unit.strip(" -•\t")
        for unit in re.split(r"\n|(?<=[.!?])\s+(?=\S)", cleaned)
        if unit.strip()
    ]


def trim_left(text: str, max_words: int = MAX_SPAN_WORDS) -> str:
    clause = re.split(r"[;:]", text)[-1]
    return " ".join(clause.strip(" ,;- ").split()[-max_words:])


def trim_right(text: str, max_words: int = MAX_SPAN_WORDS) -> str:
    clause = re.split(r"[;:]", text)[0]
    return " ".join(clause.strip(" ,;- ").split()[:max_words])


def clean_span(value: str) -> str:
    cleaned = normalize_source_text(value)
    cleaned = re.sub(
        r"^(?:chapter|section|article|figure|table|appendix|annex|"
        r"chapitre|partie|figure|tableau|annexe)\s+[A-Za-z0-9.:-]+\s*",
        "",
        cleaned,
        flags=re.IGNORECASE,
    )
    cleaned = re.sub(r"^(?:and|or|et|ou)\s+", "", cleaned, flags=re.IGNORECASE)
    cleaned = re.sub(r"\s+(?:to|de|à|pour)$", "", cleaned, flags=re.IGNORECASE)
    return re.sub(r"\s+", " ", cleaned).strip(" ,;:-")[:500]


def is_low_information_span(value: str) -> bool:
    tokens = re.findall(r"[A-Za-zÀ-ÖØ-öø-ÿ0-9]+", str(value).lower())
    if len(tokens) < MIN_SPAN_WORDS or len(tokens) > MAX_SPAN_WORDS:
        return True
    if content_token_ratio(value) < 0.46:
        return True
    artifact_hits = sum(token in CAUSAL_ARTIFACT_TOKENS for token in tokens)
    return artifact_hits >= 3


def is_plausible_cue_usage(sentence: str, match: re.Match, relation_type: str) -> bool:
    cue = re.sub(r"\s+", " ", match.group(0).lower()).strip()
    before_tokens = re.findall(
        r"[A-Za-zÀ-ÖØ-öø-ÿ]+",
        sentence[:match.start()].lower(),
    )
    after_tokens = re.findall(
        r"[A-Za-zÀ-ÖØ-öø-ÿ]+",
        sentence[match.end():].lower(),
    )
    previous_word = before_tokens[-1] if before_tokens else ""
    next_word = after_tokens[0] if after_tokens else ""
    previous_character = sentence[match.start() - 1] if match.start() > 0 else ""
    next_character = sentence[match.end()] if match.end() < len(sentence) else ""

    # Hyphenated participles such as AI-amplified or AI-generated are
    # modifiers, not standalone causal predicates.
    if previous_character == "-" or next_character == "-":
        return False

    if cue in {"cause", "risk", "support", "need", "limit", "increase", "decrease"}:
        if previous_word in NOUN_PREDECESSORS:
            return False

    if cue == "cause" and previous_word in {"a", "the", "rather", "root", "main"}:
        return False
    if cue in {"support", "supporting"}:
        if previous_word in {
            "provide", "provides", "provided", "providing", "obtain", "obtains",
            "obtained", "obtaining", "receive", "receives", "received",
            "receiving", "funding", "scientific", "pillar", "competency",
            "section", "chapter", "objective", "action", "and", "or",
            "understanding",
        }:
            return False
        if next_word in SUPPORT_NOUN_FOLLOWERS:
            return False
    if cue in {"risk", "risks"}:
        if previous_word in {
            "a", "an", "the", "other", "potential", "greatest", "significant",
            "various", "several", "these", "those", "related", "associated",
        }:
            return False
        if next_word in {
            "is", "are", "was", "were", "that", "to", "across", "related",
            "associated", "identified", "include", "including",
        }:
            return False
    if cue == "raised" and next_word == "by":
        return False

    if cue in {
        "improved", "enhanced", "strengthened", "increased", "reduced",
        "limited", "generated", "created",
    } and previous_word in {
        "a", "an", "the", "and", "or", "new", "better", "more", "less",
        "with", "for", "of", "is", "are", "was", "were", "be", "been",
        "being", "remains", "remained",
    }:
        return False
    if cue.endswith("ing") and previous_word in {
        "in", "for", "of", "by", "through", "while", "after", "before",
        "during", "as", "including",
    }:
        return False
    if cue.startswith("contribut") and previous_word in {"why", "for", "by", "of"}:
        return False

    if cue in {"facilité", "facilités"}:
        return False

    if cue.startswith("aide à") and previous_word in {"l", "la", "une", "cette", "son"}:
        return False

    before_lowered = sentence[:match.start()].lower()
    if cue.endswith("ing") and (
        "types of" in before_lowered
        or re.search(r"\b(?:include|includes|included|including)\b", before_lowered)
    ):
        return False

    if next_word in {
        "and", "or", "but", "not", "et", "ou", "mais", "for", "of",
        "from", "with",
    }:
        return False
    return True


def span_quality(cause: str, effect: str, cue_weight: float) -> float:
    left = len(cause.split())
    right = len(effect.split())
    completeness = min(1.0, (left + right) / 28.0)
    balance = min(left, right) / max(left, right)
    content = min(content_token_ratio(cause), content_token_ratio(effect))
    length_penalty = 1.0 if max(left, right) <= 40 else 0.85
    return float(
        cue_weight
        * length_penalty
        * (0.48 * completeness + 0.30 * balance + 0.22 * content)
    )


def extract_frames(frame: pd.DataFrame, corpus_type: str) -> pd.DataFrame:
    candidates = []
    for _, document in frame.iterrows():
        for sentence_index, sentence in [(int(document["sentence_index"]), document["clean_sentence"])]:
            if is_source_residue(sentence):
                continue
            for cue_pattern, relation_type, orientation, cue_weight in FRAME_CUES:
                for match in cue_pattern.finditer(sentence):
                    if not is_plausible_cue_usage(sentence, match, relation_type):
                        continue
                    left = clean_span(trim_left(sentence[:match.start()]))
                    right = clean_span(trim_right(sentence[match.end():]))
                    cause, effect = (right, left) if orientation == "reverse" else (left, right)
                    if is_low_information_span(cause) or is_low_information_span(effect):
                        continue
                    quality = span_quality(cause, effect, cue_weight)
                    if quality < MIN_FRAME_QUALITY:
                        continue
                    candidates.append({
                        "corpus_type": corpus_type,
                        "doc_id": document["doc_id"],
                        "chunk_id": document["chunk_id"],
                        "sentence_id": document["sentence_id"],
                        "chunk_index": int(document["chunk_index"]),
                        "sentence_index": int(sentence_index),
                        "filename": document["filename"],
                        "source_file": document["source_file"],
                        "country": document["analysis_country"],
                        "heading_context": document.get("heading_context", ""),
                        "frame_sentence": normalize_source_text(sentence),
                        "context_window": document.get("context_window", ""),
                        "cause_span": cause,
                        "relation_type": relation_type,
                        "relation_cue": match.group(0),
                        "cue_orientation": orientation,
                        "effect_span": effect,
                        "structured_frame": f"{cause} [{relation_type}] {effect}",
                        "frame_quality": quality,
                        "frame_signature": (
                            f"{normalise_span(cause)}|{relation_type}|"
                            f"{normalise_span(effect)}"
                        ),
                        "sentence_signature": normalise_span(sentence),
                        "extraction_method": "quality_scored_local_causal_frame",
                    })

    result = pd.DataFrame(candidates)
    if result.empty:
        return result

    result = result.sort_values("frame_quality", ascending=False)

    # Overlapping chunks often repeat the same sentence. Keep one frame per
    # document/signature and one strongest frame per relation family in the
    # same document sentence.
    result = result.drop_duplicates(
        subset=["corpus_type", "doc_id", "frame_signature"],
        keep="first",
    )
    result = result.drop_duplicates(
        subset=["corpus_type", "doc_id", "sentence_signature", "relation_type"],
        keep="first",
    )
    result = result.sort_values(
        ["doc_id", "chunk_index", "sentence_index", "relation_type"]
    ).reset_index(drop=True)
    result.insert(
        0,
        "frame_id",
        [f"{corpus_type}_frame_{index:05d}" for index in range(len(result))],
    )

    source_counts = result.groupby(
        ["corpus_type", "source_file"]
    )["frame_id"].transform("count")
    result["source_balanced_weight"] = 1.0 / source_counts.clip(lower=1)
    return result


policy_frames = extract_frames(policy_chunks, "policy")
sentiment_frames = extract_frames(sentiment_chunks, "sentiment")
synthetic_frames = extract_frames(synthetic_chunks, "synthetic_sentiment")

if policy_frames.empty or sentiment_frames.empty:
    raise ValueError("Independent frame extraction produced no empirical frames.")

pd.concat([policy_frames, sentiment_frames], ignore_index=True).to_csv(
    OUTPUT_DIR / "causal_frame_extraction_audit.csv",
    index=False,
)

extraction_summary = pd.concat(
    [
        policy_frames.assign(corpus="policy"),
        sentiment_frames.assign(corpus="sentiment"),
        synthetic_frames.assign(corpus="synthetic_sentiment"),
    ],
    ignore_index=True,
).groupby(["corpus", "relation_type"], as_index=False).agg(
    frames=("frame_id", "count"),
    documents=("doc_id", "nunique"),
    sources=("source_file", "nunique"),
    mean_frame_quality=("frame_quality", "mean"),
)
extraction_summary.to_csv(
    OUTPUT_DIR / "causal_frame_extraction_summary.csv",
    index=False,
)

frame_validation_rows = []
for corpus_name, extracted in [
    ("policy", policy_frames),
    ("sentiment", sentiment_frames),
    ("synthetic_sentiment", synthetic_frames),
]:
    residue_mask = extracted["frame_sentence"].fillna("").astype(str).str.contains(
        r"https?://|h\s+tt\s+ps|www\.|doi\.org|"
        r"[\w.+-]+@[\w.-]+\.[A-Za-z]{2,}|\|",
        case=False,
        regex=True,
    )
    duplicate_rows = extracted.duplicated(
        ["corpus_type", "doc_id", "frame_signature"]
    )
    frame_validation_rows.append({
        "corpus": corpus_name,
        "frames": len(extracted),
        "minimum_frame_quality": float(extracted["frame_quality"].min())
        if not extracted.empty else np.nan,
        "mean_frame_quality": float(extracted["frame_quality"].mean())
        if not extracted.empty else np.nan,
        "source_residue_rows": int(residue_mask.sum()),
        "within_document_duplicate_rows": int(duplicate_rows.sum()),
        "maximum_sentence_words": int(
            extracted["frame_sentence"].fillna("").str.split().str.len().max()
        ) if not extracted.empty else 0,
    })

frame_validation = pd.DataFrame(frame_validation_rows)
frame_validation.to_csv(
    OUTPUT_DIR / "causal_frame_quality_validation.csv",
    index=False,
)
if (
    frame_validation["source_residue_rows"].gt(0).any()
    or frame_validation["within_document_duplicate_rows"].gt(0).any()
    or frame_validation["minimum_frame_quality"].lt(MIN_FRAME_QUALITY).any()
):
    raise ValueError(
        "Frame quality validation failed. See causal_frame_quality_validation.csv."
    )

print("Policy frames:", len(policy_frames))
print("Sentiment frames:", len(sentiment_frames))
print("Synthetic frames:", len(synthetic_frames))
print(extraction_summary.to_string(index=False))


Policy frames: 3262
Sentiment frames: 723
Synthetic frames: 694
             corpus          relation_type  frames  documents  sources  mean_frame_quality
             policy    causes_or_increases     929         49       49            0.732074
             policy    enables_or_supports    1474         50       50            0.697599
             policy    expected_to_improve     299         42       42            0.676936
             policy    reduces_or_prevents     181         40       40            0.721773
             policy requires_or_depends_on     259         40       40            0.681026
             policy     risks_or_threatens     120         32       32            0.718702
          sentiment    causes_or_increases     306         15       15            0.747666
          sentiment    enables_or_supports     251         16       16            0.707658
          sentiment    expected_to_improve      51         12       12            0.727602
          sentiment    red

## Construct phrase-level directed edges
Aggregate extracted frames by normalized cause, relation type, and effect.


In [4]:
# Build a directed edge list from the original extracted cause/effect spans.
# Unlike a topic graph, each node is a normalized textual span.
def normalized_node(value: str) -> str:
    return re.sub(r"\s+", " ", re.sub(r"[^a-zà-öø-ÿ0-9]+", " ", str(value).lower())).strip()

for frame in (policy_frames, sentiment_frames, synthetic_frames):
    frame["cause_node"] = frame["cause_span"].map(normalized_node)
    frame["effect_node"] = frame["effect_span"].map(normalized_node)

empirical_edges = pd.concat([policy_frames, sentiment_frames], ignore_index=True)
empirical_edges["edge_key"] = (
    empirical_edges["cause_node"] + " | " + empirical_edges["relation_type"]
    + " | " + empirical_edges["effect_node"]
)
edge_support = empirical_edges.groupby(
    ["corpus_type", "cause_node", "relation_type", "effect_node"], as_index=False
).agg(frames=("frame_id", "count"), source_count=("source_file", "nunique"),
      mean_quality=("frame_quality", "mean"))
edge_support.to_csv(OUTPUT_DIR / "global_directed_relation_edges.csv", index=False)
print("Empirical frames:", len(empirical_edges))
print("Distinct directed edges:", len(edge_support))


Empirical frames: 3985
Distinct directed edges: 3909


## Compare relation distributions
Compute source-balanced relation frequencies, Jensen–Shannon divergence, and document bootstrap variation.


In [5]:
# Compare global relation-family distributions using the original Jensen-Shannon statistic.
EPSILON = 1e-8
relation_names = sorted(set(policy_frames["relation_type"]).union(sentiment_frames["relation_type"]))

def relation_distribution(frame: pd.DataFrame) -> pd.Series:
    weights = frame.groupby("relation_type")["source_balanced_weight"].sum().reindex(relation_names, fill_value=0)
    return weights.div(weights.sum()) if weights.sum() else weights

policy_distribution = relation_distribution(policy_frames)
sentiment_distribution = relation_distribution(sentiment_frames)
relation_comparison = pd.DataFrame({
    "relation_type": relation_names,
    "policy_share": policy_distribution.to_numpy(),
    "commentary_share": sentiment_distribution.to_numpy(),
})
relation_comparison["signed_difference"] = (
    relation_comparison["policy_share"] - relation_comparison["commentary_share"]
)
relation_comparison["absolute_difference"] = relation_comparison["signed_difference"].abs()
relation_comparison.to_csv(OUTPUT_DIR / "global_relation_family_comparison.csv", index=False)
js_distance = float(jensenshannon(
    policy_distribution.to_numpy() + EPSILON,
    sentiment_distribution.to_numpy() + EPSILON, base=2,
))

# Re-sample by source document to describe uncertainty in the global distribution gap.
rng = np.random.default_rng(RANDOM_STATE)
policy_docs = policy_frames["doc_id"].drop_duplicates().to_numpy()
sentiment_docs = sentiment_frames["doc_id"].drop_duplicates().to_numpy()
bootstrap_rows = []
for repeat in range(50):
    sampled_groups = []
    for name, frame, sources in (("policy", policy_frames, policy_docs),
                                 ("commentary", sentiment_frames, sentiment_docs)):
        draws = rng.choice(sources, size=len(sources), replace=True)
        # Each appearance contributes one document's frame weights.
        counts = pd.Series(draws).value_counts()
        subframe = frame[frame["doc_id"].isin(counts.index)].copy()
        subframe["source_balanced_weight"] *= subframe["doc_id"].map(counts)
        sampled_groups.append(relation_distribution(subframe))
    bootstrap_rows.append({"repeat": repeat, "js_divergence": float(
        jensenshannon(sampled_groups[0].to_numpy() + EPSILON,
                       sampled_groups[1].to_numpy() + EPSILON, base=2) ** 2)})
bootstrap_df = pd.DataFrame(bootstrap_rows)
bootstrap_df.to_csv(OUTPUT_DIR / "global_relation_bootstrap.csv", index=False)
run_metrics = pd.DataFrame([{"js_divergence": js_distance ** 2,
    "bootstrap_mean": float(bootstrap_df["js_divergence"].mean()),
    "bootstrap_low_95": float(bootstrap_df["js_divergence"].quantile(.025)),
    "bootstrap_high_95": float(bootstrap_df["js_divergence"].quantile(.975)),
    "policy_frames": len(policy_frames), "commentary_frames": len(sentiment_frames)}])
run_metrics.to_csv(OUTPUT_DIR / "global_relation_metrics.csv", index=False)
print(run_metrics.to_string(index=False))


 js_divergence  bootstrap_mean  bootstrap_low_95  bootstrap_high_95  policy_frames  commentary_frames
      0.021073        0.031855          0.014989           0.055811           3262                723


## Inspect observed relation evidence
Keep quoted evidence associated with every directed edge.


In [6]:
# Rank globally observed directed relations and retain inspectable examples.
edge_evidence = edge_support.sort_values(["source_count", "frames", "mean_quality"], ascending=False).head(60)
examples = empirical_edges.merge(
    edge_evidence[["corpus_type", "cause_node", "relation_type", "effect_node"]],
    on=["corpus_type", "cause_node", "relation_type", "effect_node"], how="inner",
).sort_values("frame_quality", ascending=False)
examples = examples.drop_duplicates(
    ["corpus_type", "cause_node", "relation_type", "effect_node"]
).head(60)
examples.to_csv(OUTPUT_DIR / "global_relation_evidence.csv", index=False)
print(examples[["corpus_type", "relation_type", "source_file", "frame_quality"]].head(12))


    corpus_type        relation_type                                        source_file  frame_quality
85    sentiment  reduces_or_prevents  markdown/cleaned/sentiment/Irfan_2023_Sentimen...       0.986250
29       policy  causes_or_increases  markdown/cleaned/policy/AI competency framewor...       0.961429
96    sentiment  causes_or_increases  markdown/cleaned/sentiment/3RemittancesReview-...       0.958328
69    sentiment  causes_or_increases  markdown/cleaned/sentiment/Irfan_2023_Sentimen...       0.947059
113   sentiment  causes_or_increases  markdown/cleaned/sentiment/3RemittancesReview-...       0.946667
40       policy  enables_or_supports  markdown/cleaned/policy/AI competency framewor...       0.934167
27       policy  causes_or_increases  markdown/cleaned/policy/australia/AI systems i...       0.914524
34       policy  causes_or_increases  markdown/cleaned/policy/AI competency framewor...       0.886190
118   sentiment  causes_or_increases  markdown/cleaned/sentiment/3Remitta

## Synthetic sensitivity
Measure shifts separately from observed public commentary.


In [7]:
# Synthetic frames are reported only as a perturbation, never as empirical evidence.
if not synthetic_frames.empty:
    synthetic_distribution = relation_distribution(synthetic_frames)
    synthetic_comparison = pd.DataFrame({
        "relation_type": relation_names,
        "empirical_commentary_share": sentiment_distribution.to_numpy(),
        "synthetic_share": synthetic_distribution.to_numpy(),
    })
    synthetic_comparison["difference"] = (
        synthetic_comparison["synthetic_share"] - synthetic_comparison["empirical_commentary_share"]
    )
    synthetic_comparison.to_csv(OUTPUT_DIR / "synthetic_relation_sensitivity.csv", index=False)
    print(synthetic_comparison.to_string(index=False))


         relation_type  empirical_commentary_share  synthetic_share  difference
   causes_or_increases                    0.385577         0.379002   -0.006575
   enables_or_supports                    0.352580         0.193056   -0.159524
   expected_to_improve                    0.066513         0.032231   -0.034282
   reduces_or_prevents                    0.077397         0.018543   -0.058854
requires_or_depends_on                    0.072324         0.348856    0.276532
    risks_or_threatens                    0.045610         0.028312   -0.017298


## Global relation figures
Render static diagnostic figures compatible with VS Code virtual machines.


In [8]:
# Draw two global relation-family summaries without browser-based 3D graphics.
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
y = np.arange(len(relation_names))
axes[0].barh(y - .18, relation_comparison["policy_share"], height=.36, label="Policy")
axes[0].barh(y + .18, relation_comparison["commentary_share"], height=.36, label="Commentary")
axes[0].set_yticks(y, relation_names)
axes[0].set_title("Global causal-relation distribution")
axes[0].set_xlabel("Weighted relation share")
axes[0].legend()
axes[1].barh(relation_comparison["relation_type"], relation_comparison["signed_difference"])
axes[1].axvline(0, color="black", linewidth=.8)
axes[1].set_title("Policy minus commentary")
axes[1].set_xlabel("Signed frequency difference")
fig.tight_layout()
save_verified_png(fig, IMG_DIR / "global_relation_patterns.png")


Saved verified PNG: /home/mouelhis/Private/github/new_dsac/cnlp/causal_semantics/figures/relation_graphs/global_relation_patterns.png size=(2053, 783)


## Record assumptions and limitations
Frequency differences cannot establish real-world cause and effect.


In [9]:
# Summarize method and scope without attributing causality to text-derived signals.
summary = pd.DataFrame([{
    "method": "global_directed_causal_language_graph",
    "scope": "pooled_global", "topic_dependencies": 0,
    "source_segments": CHUNKS_ALL.name, "policy_frames": len(policy_frames),
    "commentary_frames": len(sentiment_frames), "synthetic_frames": len(synthetic_frames),
    "distinct_empirical_edges": len(edge_support),
    "relation_js_divergence": js_distance ** 2,
    "interpretation": "Patterns are extracted statements, not validated causal effects",
}])
summary.to_csv(OUTPUT_DIR / "run_summary.csv", index=False)
print(summary.to_string(index=False))


                               method         scope  topic_dependencies              source_segments  policy_frames  commentary_frames  synthetic_frames  distinct_empirical_edges  relation_js_divergence                                                  interpretation
global_directed_causal_language_graph pooled_global                   0 policy_feedback_segments.csv           3262                723               694                      3909                0.021073 Patterns are extracted statements, not validated causal effects
